In [ ]:
import os, re, math, glob
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

sample_dir = "/content/sample_ablation"      # 确认这次训练确实还是存到这里
run_tag = "run_ablation"                    # 【新增】给这次训练起个标签，避免覆盖上次结果

batch = 32          # 【核对】必须和这次训练脚本里的 --batch 一致
img_size = 64        # 【核对】必须和这次训练脚本里的 --size 一致
ncols = int(math.sqrt(batch))
padding = 2

def load_img(path):
    return transforms.ToTensor()(Image.open(path).convert('RGB'))

def split_grid(grid_tensor, batch_size, ncols, tile_size, padding=2):
    nrows = math.ceil(batch_size / ncols)
    tiles = []
    for idx in range(batch_size):
        r, c = idx // ncols, idx % ncols
        y0 = padding + r * (tile_size + padding)
        x0 = padding + c * (tile_size + padding)
        tiles.append(grid_tensor[:, y0:y0+tile_size, x0:x0+tile_size])
    return torch.stack(tiles)

def psnr(a, b, max_val=1.0):
    mse = torch.mean((a - b) ** 2, dim=[1,2,3])
    return 10 * torch.log10(max_val**2 / mse.clamp(min=1e-10))

def ssim_single(a, b, C1=0.01**2, C2=0.03**2):
    mu_a, mu_b = a.mean(), b.mean()
    var_a, var_b = a.var(), b.var()
    cov = ((a - mu_a) * (b - mu_b)).mean()
    return ((2*mu_a*mu_b + C1) * (2*cov + C2)) / ((mu_a**2 + mu_b**2 + C1) * (var_a + var_b + C2))

orig_grid = load_img(os.path.join(sample_dir, "sample.png"))
orig_tiles = split_grid(orig_grid, batch, ncols, img_size, padding)

rec_files = sorted(
    glob.glob(os.path.join(sample_dir, "[0-9]" * 6 + ".png")),
    key=lambda p: int(re.findall(r"\d+", os.path.basename(p))[0])
)

steps, psnr_list, ssim_list = [], [], []
for f in rec_files:
    step = int(re.findall(r"\d+", os.path.basename(f))[0])
    rec_grid = load_img(f)
    rec_tiles = split_grid(rec_grid, batch, ncols, img_size, padding)
    p = psnr(orig_tiles, rec_tiles).mean().item()
    s = np.mean([ssim_single(orig_tiles[i], rec_tiles[i]).item() for i in range(batch)])
    steps.append(step); psnr_list.append(p); ssim_list.append(s)
    print(f"step {step:>6}: PSNR={p:.2f} dB, SSIM={s:.4f}")

# 【新增】保存原始数据，方便和上一轮训练对比
np.savez(f"/content/sample_ablation/metrics_ablation{run_tag}.npz",
         steps=np.array(steps), psnr=np.array(psnr_list), ssim=np.array(ssim_list))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(steps, psnr_list, marker='o')
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title(f"PSNR over training ({run_tag})")   # 【改】标题加标签
axes[0].grid(alpha=0.3)

axes[1].plot(steps, ssim_list, marker='o', color='orange')
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("SSIM")
axes[1].set_title(f"SSIM over training ({run_tag})")    # 【改】标题加标签
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"/content/sample/psnr_ssim_curve_ablation{run_tag}.png", dpi=150)  # 【改】文件名加标签
plt.show()

In [ ]:
with torch.no_grad():
    real = next(dataloader)['image'].to(args.device)
    h = model.encoder(real)
    p = model.quant_conv(h)   # 这就是Sigmoid后的概率，形状 (B, vq_embed_dim, H', W')

    eps = 1e-8
    p_clamped = p.clamp(eps, 1 - eps)
    binary_entropy = -(p_clamped * torch.log2(p_clamped) + (1 - p_clamped) * torch.log2(1 - p_clamped))
    bits_per_image = binary_entropy.sum(dim=[1,2,3])  # 每张图的真实信息熵（理论最优编码下的bit数）
    bpp_actual = bits_per_image / (args.size * args.size)

    print(f"latent形状: {p.shape}")
    print(f"平均bpp(基于真实熵): {bpp_actual.mean().item():.5f}")
    print(f"简单计数上限bpp: {p.shape[1]*p.shape[2]*p.shape[3] / (args.size**2):.5f}")

In [ ]:
with torch.no_grad():
    real = next(dataloader)['image'].to(args.device)
    h = model.encoder(real)
    p = model.quant_conv(h)   # (B, 32, 4, 4)

    # 展平成 (32, B*4*4)，看各通道间的相关性
    p_flat = p.permute(1,0,2,3).reshape(p.shape[1], -1)
    corr = torch.corrcoef(p_flat)

    print(f"latent形状: {p.shape}")
    print(f"通道间平均绝对相关系数(排除自相关): "
          f"{(corr.abs().sum() - corr.shape[0]) / (corr.shape[0]**2 - corr.shape[0]):.4f}")

In [ ]:
import os, glob, math
from tqdm import tqdm
from torch import nn, optim
import torch, argparse
import sys
sys.path.append('/content')
from vq_model import VQModel
from lossers.lpips import LPIPS
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from torchvision.transforms import functional as tvf
from discriminator import NLayerDiscriminator
from PIL import Image
from lossers.gan import hinge_d_loss as d_loss_fn, vanilla_g_loss as g_loss_fn

class LocalImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, exts=('jpg', 'jpeg', 'png')):
        self.transform = transform
        self.paths = []
        for ext in exts:
            self.paths += glob.glob(os.path.join(root_dir, '**', f'*.{ext}'), recursive=True)
            self.paths += glob.glob(os.path.join(root_dir, '**', f'*.{ext.upper()}'), recursive=True)
        self.paths = sorted(set(self.paths))
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform is not None: img = self.transform(img)
        return {'image': img}

def sample_data(loader):
    while True:
        for batch in loader:
            yield batch

# 用之前训练时一样的参数（确保 size/batch 等一致，不然模型结构或数据形状对不上）
parser = argparse.ArgumentParser()
parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--dataset", type=str, default="/content/D-Fire")
parser.add_argument("--cache_dir", type=str, default="./.cache")
parser.add_argument("--batch", type=int, default=32)
parser.add_argument("--size", type=int, default=64)
args, unknown = parser.parse_known_args()

# 重建模型结构（此时是随机初始化，还没加载权重）
model = VQModel().to(args.device)
lpips = LPIPS(net='vgg', cache_dir=args.cache_dir).to(args.device)
discriminator = NLayerDiscriminator().to(args.device)
vq_optim = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.0, 0.999))
d_optim = torch.optim.Adam(discriminator.parameters(), lr=1e-3, betas=(0.0, 0.999))

# 重建dataloader
to_tensor = transforms.Compose([
    transforms.Resize(args.size, interpolation=tvf.InterpolationMode.LANCZOS),
    transforms.CenterCrop(args.size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5), inplace=True),
])
dataset = LocalImageDataset(args.dataset, transform=to_tensor)
dataloader = DataLoader(dataset, batch_size=args.batch, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
dataloader = sample_data(dataloader)

def requires_grad(m, flag=True):
    for p in m.parameters(): p.requires_grad = flag